# AXI: can the host say where the piece goes before the move? A free check on the saved ARC-AGI-3 runs (arms K, A, B and C)
**What this is.** The saved moves of Tests 24, 25 and 26 are played back into fresh copies of the four public ARC-AGI-3 games (the replay still checks every saved flag), and for every move the host predicts, from the earlier moves of the same level only, where the piece will go. It is scored against the move that was played. Page: `docs/FORESEE_CHECK.md` (8 reading lines, locked before this runs). Comes from pillar III of Rajnish Choubey's RFC-001 (a candidate is checked on a shadow before it is committed); how it is read in game terms is for Sir to confirm.

**It calls no model, needs no key, no Google Drive and costs nothing.** It plays nothing new.

**What it does not show:** that a path-finder built on this would clear a level, or what the goal of a level is.

### Cell 1: get the code and the saved results from GitHub, and install the ARC toolkit (about 1 minute)

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
!pip -q install arc-agi arcengine 2>&1 | tail -2
import os, sys
sys.path.insert(0, "/content/axi/python")
assert os.path.exists("/content/axi/python/axi/experiments/foresee.py"), "the check is not on GitHub yet"
for m in [m for m in sys.modules if m == "axi" or m.startswith("axi.")]: del sys.modules[m]      # a second run in the same session must not keep the code of the first
!git -C /content/axi log --oneline -1
print("code ok")

### Cell 2: the games (the toolkit takes an anonymous key by itself)

In [ ]:
import json, arc_agi
from axi.experiments import gemini_arc as G
from axi.experiments import ledger_replay as LR
from axi.experiments import foresee as F
ARC = arc_agi.Arcade()
def make_env(game): return ARC.make(game, seed=G.GAME_SEED)
RES = "/content/axi/docs/results/"
FILES = [("axi_gemini_arc_default.json", "K"), ("axi_gemini_arc_default.json", "A"), ("axi_gemini_arc_b_default.json", "B"), ("axi_gemini_arc_c_default.json", "C")]
print("ready:", [f"{f} arm {a}" for f, a in FILES])

### Cell 3: replay each arm and predict every move from the moves before it. If a line says `ReplayMismatch`, the saved moves did not replay into the game (copy the whole message)

In [ ]:
per_arm, per_game, ok = {}, {}, {}
for f, arm in FILES:
    print("=" * 100); print("ARM", arm, "of", f); print("=" * 100)
    try:
        s = LR.run_file(RES + f, arm, make_env)
        pg = F.from_session(s); per_arm[arm] = pg; ok[arm] = True
        print(F.text(pg))
        for g, c in pg.items(): per_game[g] = F.combine([per_game.get(g, {}), c])
    except Exception as e:
        print(type(e).__name__, ":", e); ok[arm] = False; per_arm[arm] = {"error": f"{type(e).__name__}: {e}"}

### Cell 4: the reading. The eight lines of the page, over the four arms together

In [ ]:
checks = F.verdict(per_game if all(ok.values()) else None, all(ok.values()) if ok else None)
print("ALL FOUR ARMS TOGETHER"); print(F.text(per_game))
print()
for i, t, x, v in checks: print(i, t, "|", {True: "yes", False: "NO", None: "not read"}[v], "|", x)
print("headline:", F.headline(checks))

### Cell 5: save the numbers and download the file. Send me the printout above and this file

In [ ]:
from google.colab import files
json.dump(dict(per_arm=per_arm, per_game=per_game, replays=ok, checks=checks), open("foresee_check_all.json", "w"), indent=1, default=str)
files.download("foresee_check_all.json")

### Done
Nothing here changed anything on GitHub or called any model.